In [2]:
mol_dir = "./mols"
basis_sets_dir = "./basis-sets"

particle_properties_file = "particle-properties.json"

In [3]:
e_basis_set = "def2-SVP"
n_basis_set = "DZSPN"

mol_name = "LiH"

In [37]:
import numpy as np
import json
import itertools

from scipy.sparse import coo_matrix, csr_matrix

from pyscf import gto, scf
from pyscf.lo import orth

from gbasis.wrappers import from_pyscf
from gbasis.parsers import parse_nwchem
from gbasis.parsers import make_contractions

from gbasis.integrals.overlap import overlap_integral
from gbasis.integrals.kinetic_energy import kinetic_energy_integral
from gbasis.integrals.electron_repulsion import electron_repulsion_integral

# Load properties of all possible particles (spin, fermion/boson, mass, charge, etc)
with open(particle_properties_file, "r") as file:
    particle_properties = json.load(file)

# Build molecule for PySCF
mol = gto.Mole()
mol.atom = mol_dir + '/' + mol_name + '.xyz'
mol.basis = e_basis_set
mol.build()

mol_zs = mol.atom_charges()
mol_symbs = [mol.atom_symbol(i) for i in range(mol.natm)] # Atomic symbols
mol_coords = mol.atom_coords()

# Run restricted Hartree Fock to get better orbitals (I might truncate the highest energy ones)
hf = scf.RHF(mol).run() # TODO: apparently there might be better choices of orbital to allow for truncations (FNO). look into?

# Load basis dictionary (atomic orbitals) for nuclear orbitals
n_basis_dict = parse_nwchem(basis_sets_dir + '/nuclear/' + n_basis_set + '.nw')

# Construct a dictionary of all the particle types that will be in our calculation, along with their orbitals and info like spin.
# Note: we will use the order of the dictionary. Python 3.7+ guarantees when we iterate, the dictionary will be ordered according to when the elements were added.
particles = {}

for i in range(mol.natm):
    symb = mol.atom_symbol(i)
    # If this is a particle type (nucleus) we haven't seen before
    if symb not in particles:
        # Add it to particle list
        particles[symb] = {}
        particles[symb]['coords'] = []
        particles[symb]['count'] = 0

    # Add its coordinates to the list
    particles[symb]['coords'].append(mol_coords[i])

    particles[symb]['count'] += 1

for symb in particles:
    # GBasis wants coords as numpy array
    particles[symb]['coords'] = np.array(particles[symb]['coords'])

    # Construct a basis w/ GBasis for each of the nuclear particles
    particles[symb]['basis'] = make_contractions(n_basis_dict, # Basis of (nuclear) AOs to use
                                                 [symb] * particles[symb]['count'], # Types of atoms (all the same)
                                                 particles[symb]['coords'], # Coordinates
                                                 coord_types='cartesian')

    # Transform to orthonormal orbitals
    overlap = overlap_integral(particles[symb]['basis'])
    particles[symb]['transform'] = orth.lowdin(overlap) # Symmetric orthonormalization of AOs

    # Number of spatial orbitals
    particles[symb]['no_spatial_orbitals'] = overlap.shape[0]

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['properties'] = particle_properties[symb]

# Add electrons to our particle list
particles['e'] = {}
particles['e']['basis'] = from_pyscf(mol) # Gbasis set of GTOs (gaussian type orbitals) for electronic particles
particles['e']['transform'] = hf.mo_coeff.T # transform to MOs that will be used for calculation
particles['e']['count'] = mol.nelectron  # Get number of electrons from PySCF, truncate off 10
particles['e']['no_spatial_orbitals'] = hf.mo_coeff.shape[0] - 7

# Retrieve some important properties on each particle (spin, fermion/boson, mass, charge)
for symb in particles:
    particles[symb]['properties'] = particle_properties[symb]

    # Number of spin orbitals, since we now have the particle's spin
    particles[symb]['no_spin_orbitals'] = particles[symb]['no_spatial_orbitals'] * particles[symb]['properties']['spin']

# Amount of particles we have
particle_types = len(particles)

# List of particle names for easy indexing
particle_names = [symb for symb in particles]

# For full FCI, we will construct the Hamiltonian in the subspace of all states with only the correct particle numbers
# A basis for this space is constructed from N-particle determinants/permanents of the one-particle basis states for correct N

# Total number of states in this space
total_states = 1

# Number of permanents/determinants for each particle.
no_states = []

# For the Hamiltonian matrix, we will index the states as follows:
# Let N_i be the number of states for the i-th particle
# The state formed from the c_0 - th particle 1 permanent/determinant, c_1 - th particle 2 perminant/determinant, etc (c_i's zero indexed)
# Will get the index (c_0) + (c_1 * N_0) + (c_2 * N_1 * N_0) + (c_3 * N_2 * N_1 * N_0) + ...

# This works basically like a numeral base system where each digit has a different base (each digit is a particle).
# This array will contain the bases for each particle: [1, N_0, N_1 * N_0, ...]
bases = []

for symb in particles:
    particles[symb]['states'] = []

    # Construct all N-particle states for the correct N, in the forms of arrays of 1s and 0s
    for indices in itertools.combinations(range(particles[symb]['no_spin_orbitals']), particles[symb]['count']):
        array = [0] * particles[symb]['no_spin_orbitals']
        for index in indices:
            array[index] = 1

        particles[symb]['states'].append(array)

    particles[symb]['no_states'] = len(particles[symb]['states'])
    particles[symb]['base'] = total_states

    no_states.append(particles[symb]['no_states'])
    bases.append(particles[symb]['base'])
    total_states *= particles[symb]['no_states']

bases.append(total_states) # Having this extra element will be helpful in construct_1_particle_interaction

# Construct the contribution to the Hamiltonian from a single particle interaction (e.g. <e_1 | KE | e_3> where e_i are electronic basis states)
# We can have any state for the remaining particles so we will iterate over all possible determinants for every other particle

# particle_no: index in the dictionary (use particle_names for indexing)
# bra: index of the bra state in particle particle_no (this is a determinant/permanent, not a single particle state!)
# ket: inex of the ket state                          (this is a determinant/permanent, not a single particle state!)
# value: value of <bra | O_1 | ket>
def construct_1_particle_interaction(particle_no, bra, ket, value):
    # The idea here is that the states (in our huge big space) that involve this specific matrix element <bra | O_1 | ket> are those that look like
    # bra: |  ANY  |  ANY  |  ...  |  bra  |  ... |  ANY  |
    # ket: |  ANY^ |  ANY^ |  ...  |  ket  |  ... |  ANY  |
    #        ptcl 1  ptcl 2   ..   ptcl ptcl_no ..    ptcl particle_types
    # where the boxes are our choice of determinant/permanent for each particle type, and between the bra and the ket, have to be the same for all
    # particle types other than the one labeled by ptcl_no (the argument, particle_no).
    # So we will iterate over all possible choices of determinants for the other particles

    # Lists of the row and column indices and the values (will all be the same, value) in the Hamiltonian matrix.
    mtx_rows = []
    mtx_cols = []
    mtx_values = []

    # Iterate all over choices of determinant/permanent for the particles coming BEFORE ptcl_no
    for i in range(bases[particle_no]):
        # Iterate all over choices of determinant/permanent for the particles coming AFTER ptcl_no
        for j in range(int(total_states/bases[particle_no+1])):
            # Construct the indices in the Hamiltonian matrix (base convention) for these bra and ket states
            bra_idx = i + (bra * bases[particle_no]) + (j * bases[particle_no+1])
            ket_idx = i + (ket * bases[particle_no]) + (j * bases[particle_no+1])

            # Add to the list of elements
            mtx_rows.append(bra_idx)
            mtx_cols.append(ket_idx)
            mtx_values.append(value)

    return coo_matrix((mtx_values, (mtx_rows, mtx_cols)), shape=(total_states, total_states))

# We will construct the Hamiltonian matrix in this basis, one interaction at a time

h_mtx = coo_matrix((total_states,total_states)) # Create empty sparse matrix of the appropriate size

# One-body contributions (kinetic energy)
for i in range(particle_types):
    symb = particle_names[i]
    particle = particles[symb]


    # Get kinetic energy integrals
    ke_int = kinetic_energy_integral(particle['basis'], particle['transform'])

    print(symb)
    for bra in particle['states']:
        print(bra)

    #print(symb)
    #print(np.round(ke_int1e, 3))

converged SCF energy = -7.9786624829859
H
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1]
Li
[1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

In [18]:
print(bases[0])

1


In [19]:
print(total_states/bases[1])

32032.0


In [38]:
print(construct_1_particle_interaction(2, 2, 1, 4.8))

<COOrdinate sparse matrix of dtype 'float64'
	with 512 stored elements and shape (512512, 512512)>
  Coords	Values
  (1024, 512)	4.8
  (1025, 513)	4.8
  (1026, 514)	4.8
  (1027, 515)	4.8
  (1028, 516)	4.8
  (1029, 517)	4.8
  (1030, 518)	4.8
  (1031, 519)	4.8
  (1032, 520)	4.8
  (1033, 521)	4.8
  (1034, 522)	4.8
  (1035, 523)	4.8
  (1036, 524)	4.8
  (1037, 525)	4.8
  (1038, 526)	4.8
  (1039, 527)	4.8
  (1040, 528)	4.8
  (1041, 529)	4.8
  (1042, 530)	4.8
  (1043, 531)	4.8
  (1044, 532)	4.8
  (1045, 533)	4.8
  (1046, 534)	4.8
  (1047, 535)	4.8
  (1048, 536)	4.8
  :	:
  (1511, 999)	4.8
  (1512, 1000)	4.8
  (1513, 1001)	4.8
  (1514, 1002)	4.8
  (1515, 1003)	4.8
  (1516, 1004)	4.8
  (1517, 1005)	4.8
  (1518, 1006)	4.8
  (1519, 1007)	4.8
  (1520, 1008)	4.8
  (1521, 1009)	4.8
  (1522, 1010)	4.8
  (1523, 1011)	4.8
  (1524, 1012)	4.8
  (1525, 1013)	4.8
  (1526, 1014)	4.8
  (1527, 1015)	4.8
  (1528, 1016)	4.8
  (1529, 1017)	4.8
  (1530, 1018)	4.8
  (1531, 1019)	4.8
  (1532, 1020)	4.8
  (1533, 102

In [ ]:
# -> loop over all bras
#     -> loop over all interaction types
#           -> loop over all possible kets that wont give zero for this interaction type
#                h += mtx element for that interaction

# for iterating over all doubly "excited" states: iterate over all pairs of occupied states and all pairs of unoccupied states, promote the pair of occupied states to the pair of unoccupied states1

In [20]:
print(particle_properties['e']['spin'])

2


In [27]:
print(particles.keys())

dict_keys(['H', 'Li', 'e'])


In [31]:
print(particles['e']['states'][0])

[1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
